# KulitKu · Eksperimen 4 kelas dengan split tetap

Dataset: [Drive KulitKu](https://drive.google.com/drive/folders/1PPxapzkZThMkHtm7QqDFnAyDw6Z2DS_8). Kode: [branch Felix](https://github.com/Liex2810/KulitKu/tree/felix/first-try-cal-onnx-integration).

Buka di Colab, aktifkan GPU T4, lalu jalankan sel berurutan. Tambahkan shortcut folder dataset ke **My Drive** bila folder dibagikan. Isi `DATASET_ROOT` dengan lokasi folder induk yang berisi **Train**, **Val/Validation**, dan **Test** bila pencarian otomatis tidak menemukan satu folder yang cocok. Notebook mempertahankan split tersebut.

| Kelas | Train | Validation | Test | Total acuan |
|---|---:|---:|---:|---:|
| Eczema | 735 | 157 | 158 | 1050 |
| Normal | 711 | 152 | 153 | 1016 |
| Psoriasis | 580 | 124 | 124 | 828 |
| Ringworm | 679 | 145 | 146 | 970 |

Jumlah aktual dihitung sebelum dan sesudah pembersihan. Sembilan kombinasi dibandingkan pada validation tetap. Setelah pemilihan, model terbaik dikalibrasi dengan prediksi silang pada development (train + validation). Test dipakai untuk evaluasi akhir model terkalibrasi. Confidence adalah probabilitas model untuk empat kelas ini. Independensi pasien belum terverifikasi.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
from google.colab import drive

REPO = 'https://github.com/Liex2810/KulitKu.git'
BRANCH = 'felix/first-try-cal-onnx-integration'
CHECKOUT = Path('/content/KulitKu_felix_experiment')
DRIVE = Path('/content/drive/MyDrive')
DATASET_ROOT = ''  # Isi path Colab jika pencarian otomatis tidak menemukan satu kandidat.
OUTPUT_ROOT = DRIVE / 'KulitKu' / 'hasil_split_manual'
REVIEWED_NEAR_DUPLICATES = False  # True setelah pasangan lintas split ditinjau.
INSTALL_DEPENDENCIES = True

drive.mount('/content/drive')
if not CHECKOUT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, str(CHECKOUT)], check=True)
commit = subprocess.check_output(['git', '-C', str(CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip()
assert subprocess.check_output(['git', '-C', str(CHECKOUT), 'branch', '--show-current'], text=True).strip() == BRANCH
CODE_ROOT = CHECKOUT / 'machine-learning'
print('Commit:', commit)
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(CODE_ROOT / 'requirements.txt'), 'ImageHash==4.3.2'], check=True)
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
sys.path.insert(0, str(CODE_ROOT / 'src'))


In [ ]:
import tempfile
import pandas as pd
import torch
from IPython.display import display
from kulitku_ml.config import load_config, Run

CLASSES = ['Eczema', 'Normal', 'Psoriasis', 'Ringworm']
SPLIT_NAMES = {'train': 'train', 'training': 'train', 'val': 'validation',
               'valid': 'validation', 'validation': 'validation', 'test': 'test', 'testing': 'test'}
LABEL_NAMES = {label.casefold(): label for label in CLASSES}
LABEL_NAMES['psioriasis'] = 'Psoriasis'
EXPECTED = pd.DataFrame(
    [[735,157,158], [711,152,153], [580,124,124], [679,145,146]],
    index=CLASSES, columns=['train','validation','test'])

def named_dirs(root, aliases):
    found = {}
    for path in root.iterdir():
        key = aliases.get(path.name.strip().casefold())
        if key and path.is_dir():
            if key in found:
                raise ValueError(f'Dua folder mewakili {key} di {root}')
            found[key] = path
    return found

if DATASET_ROOT:
    source = Path(DATASET_ROOT)
else:
    candidates = []
    for current, dirs, _ in os.walk(DRIVE):
        path = Path(current)
        names = {SPLIT_NAMES.get(name.strip().casefold()) for name in dirs}
        if set(EXPECTED.columns) <= names:
            candidates.append(path)
            dirs.clear()
        elif len(path.relative_to(DRIVE).parts) >= 4:
            dirs.clear()
    if len(candidates) != 1:
        raise ValueError(f'Isi DATASET_ROOT dengan path folder induk dataset. Kandidat: {candidates}')
    source = candidates[0]

partition_dirs = named_dirs(source, SPLIT_NAMES)
if set(partition_dirs) != set(EXPECTED.columns):
    raise ValueError(f'Folder Train, Validation dan Test diperlukan di {source}')
class_dirs = {part: named_dirs(folder, LABEL_NAMES) for part, folder in partition_dirs.items()}
for part, dirs in class_dirs.items():
    if set(dirs) != set(CLASSES):
        raise ValueError(f'Kelas tidak lengkap di {part}: {sorted(dirs)}')

stage = Path(tempfile.mkdtemp(prefix='kulitku_manual_', dir='/content'))
extensions = {'.jpg','.jpeg','.png','.webp','.bmp','.tif','.tiff','.jfif'}
for part, dirs in class_dirs.items():
    for label, folder in dirs.items():
        destination = stage / label / part
        destination.mkdir(parents=True)
        for file in sorted(folder.rglob('*')):
            if file.is_file() and file.suffix.lower() in extensions:
                target = destination / file.relative_to(folder)
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copyfile(file, target)

cfg = load_config(CODE_ROOT / 'configs/experiment.json')
cfg['classes'] = CLASSES
cfg['paths'] = {'dataset_root': str(stage), 'output_root': str(OUTPUT_ROOT)}
cfg['split']['folds'] = 1  # Satu validation tetap untuk API training branch.
cfg['benchmark'].update(samples_per_class=8, repeats=2, warmup=3)
cfg['manual_experiment'] = {'source': str(source), 'commit': commit,
                            'protocol': 'manual_train_validation_test'}
run = Run(cfg)
print('Data:', source, '| Hasil:', run.out, '| GPU:', torch.cuda.is_available())


In [ ]:
from kulitku_ml.data import inventory
from kulitku_ml.cleaning import clean
from kulitku_ml.eda import eda
from kulitku_ml.splitting import file_hash
from kulitku_ml.preprocessing import rgb
import hashlib
import imagehash
from tqdm.auto import tqdm

before = inventory(run)
before['manual_split'] = before.relative_path.str.split('/').str[1]

def counts(frame):
    return pd.crosstab(frame.label, frame.manual_split).reindex(
        index=CLASSES, columns=EXPECTED.columns, fill_value=0)

display(pd.concat({'target': EXPECTED, 'aktual': counts(before)}, axis=1))
after = clean(run, before).copy()
after['manual_split'] = after.relative_path.str.split('/').str[1]
after = after[~after.relative_path.map(lambda p: Path(p).name.casefold().startswith('aug_'))].copy()
audit = []
for row in tqdm(after.itertuples(), total=len(after), desc='Audit hash'):
    path = run.root / row.relative_path
    image = rgb(path, cfg['cleaning']['max_pixels'])
    pixels = hashlib.sha256(str(image.size).encode() + image.tobytes()).hexdigest()
    audit.append((file_hash(path), pixels, str(imagehash.phash(image))))
after[['sha256','pixel_sha256','phash']] = pd.DataFrame(audit, index=after.index, columns=range(3))

for key in ['sha256', 'pixel_sha256']:
    by_hash = after.groupby(key)
    bad = by_hash.manual_split.transform('nunique').gt(1) | by_hash.label.transform('nunique').gt(1)
    run.table(after[bad], f'conflicts_{key}.csv')
    if bad.any():
        raise ValueError(f'Duplikat lintas split/label. Perbaiki di Drive; lihat conflicts_{key}.csv.')
after = after.sort_values('relative_path')
run.table(after[after.duplicated('pixel_sha256')], 'duplicates_removed.csv')
after = after.drop_duplicates('pixel_sha256').reset_index(drop=True)
run.table(counts(after).rename_axis('label').reset_index(), 'manual_split_counts.csv')
display(counts(after))
if (counts(after) == 0).any().any():
    raise ValueError('Setiap split harus memiliki keempat kelas setelah cleaning.')

pairs = []
records = [(r.relative_path, r.manual_split, int(r.phash, 16)) for r in after.itertuples()]
for i, (left, part, h1) in enumerate(records):
    for j in range(i + 1, len(records)):
        right, other, h2 = records[j]
        if part != other:
            distance = (h1 ^ h2).bit_count()
            if distance <= 4:
                pairs.append((left, right, distance))
near = pd.DataFrame(pairs, columns=['file_a','file_b','phash_distance'])
run.table(near, 'near_duplicates_cross_split.csv')
if len(near) and not REVIEWED_NEAR_DUPLICATES:
    display(near.head(20))
    raise ValueError('Tinjau laporan near_duplicates_cross_split.csv. Pindahkan/hapus duplikat nyata, lalu ulangi dari staging. Jika pasangan yang tersisa berbeda, set REVIEWED_NEAR_DUPLICATES=True dan ulangi sel audit.')

splits = after.assign(
    partition=after.manual_split.map({'train':'development','validation':'development','test':'test'}),
    fold=after.manual_split.map({'train':-1,'validation':0,'test':-1}),
    evaluation_eligible=True,
    split_group='pixel_' + after.pixel_sha256)
run.table(splits, 'splits.csv')
run.json({'patient_independence_verified': False, 'near_duplicate_pairs_reviewed': len(near),
          'split': 'manual'}, 'split_status.json')
display(eda(run, before[before.manual_split.eq('train')], 'train_before'))
display(eda(run, splits[splits.manual_split.eq('train')], 'train_after'))


## Ekstraksi fitur, training dan pemilihan

CNN dibekukan. Selisih Macro-F1 train–validation dihitung untuk parameter terbaik setiap kombinasi. Grafik ini mengukur gap generalisasi. Classifier klasik di eksperimen ini tidak menggunakan kurva loss per epoch.


In [ ]:
from kulitku_ml.features import extract_features
for cnn in cfg['extractors']:
    x=extract_features(run,splits,cnn)
    print(cnn,x.shape)


In [ ]:
import numpy as np
from kulitku_ml.training import train_classifier, classifier, metrics
from kulitku_ml.evaluation import validation_report
from sklearn.dummy import DummyClassifier
import matplotlib.pyplot as plt

y = splits.label.map({label: i for i, label in enumerate(CLASSES)}).to_numpy()
tr = splits.manual_split.eq('train').to_numpy()
va = splits.manual_split.eq('validation').to_numpy()
te = splits.manual_split.eq('test').to_numpy()
baseline = DummyClassifier(strategy='most_frequent').fit(np.zeros((tr.sum(),1)), y[tr])
run.json(metrics(y[va], baseline.predict(np.zeros((va.sum(),1)))), 'validation_baseline.json')
rows = []
for cnn in cfg['extractors']:
    x = np.load(run.out/'features'/cnn/'features.npy', mmap_mode='r')
    for name in cfg['classifiers']:
        result = train_classifier(run, splits, x, cnn, name)
        fitted = classifier(run, name, result['best_params']).fit(x[tr], y[tr])
        train_f1 = metrics(y[tr], fitted.predict(x[tr]))['macro_f1']
        val_f1 = result['mean_macro_f1']
        rows.append(dict(combo_id=result['combo_id'], train_macro_f1=train_f1,
                         validation_macro_f1=val_f1, gap=train_f1-val_f1))
        validation_report(run, cnn, name)
gap = pd.DataFrame(rows)
run.table(gap, 'train_validation_gap.csv')
display(gap)
ax = gap.set_index('combo_id')[['train_macro_f1','validation_macro_f1']].plot.bar(figsize=(12,5), ylim=(0,1.05))
ax.set_ylabel('Macro-F1')
run.figure(ax.figure, 'train_validation_gap.png')

from kulitku_ml.benchmarking import benchmark
display(benchmark(run, splits))
selected = json.loads((run.out/'selection.json').read_text())
print('Terpilih:', selected['combo_id'])


## Kalibrasi, ekspor dan uji akhir

Pemilihan memakai Macro-F1 validation dengan toleransi 0,01 dari skor tertinggi, lalu median latensi CPU. Setelah pilihan dikunci, kalibrasi sigmoid memakai 5 fold pada development dengan `ensemble=False`. Benchmark seleksi mengukur model native sebelum kalibrasi. Paket akhir menyimpan CNN dan classifier terkalibrasi dalam ONNX.


In [ ]:
import joblib
import onnx
import onnxruntime as ort
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
from kulitku_ml.features import restore_extractor
from kulitku_ml.preprocessing import build_transform

cnn, name = selected['cnn'], selected['classifier']
x = np.load(run.out/'features'/cnn/'features.npy', mmap_mode='r')
dev = tr | va
calibration_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=cfg['seed'])
calibrated = CalibratedClassifierCV(classifier(run, name, selected['best_params']),
                                    method='sigmoid', cv=calibration_cv, ensemble=False)
calibrated.fit(x[dev], y[dev])
model, info = restore_extractor(run, cnn)
spec = info['preprocessing']
transform = build_transform(spec)
release_dir = run.out/'onnx'/selected['combo_id']
release_dir.mkdir(parents=True, exist_ok=True)
(release_dir/'release.json').unlink(missing_ok=True)
torch.onnx.export(model, torch.zeros(1, *spec['input_size']), str(release_dir/'extractor.onnx'),
                  input_names=['images'], output_names=['features'], opset_version=17,
                  dynamic_axes={'images':{0:'batch'},'features':{0:'batch'}}, dynamo=False)
converted = convert_sklearn(calibrated, initial_types=[('features', FloatTensorType([None,x.shape[1]]))],
                             options={id(calibrated):{'zipmap':False}}, target_opset=17)
(release_dir/'classifier.onnx').write_bytes(converted.SerializeToString())
for file in ['extractor.onnx','classifier.onnx']:
    onnx.checker.check_model(str(release_dir/file))
    if not (release_dir/file).is_file():
        raise FileNotFoundError(file)

options = ort.SessionOptions()
options.intra_op_num_threads = cfg['benchmark']['threads']
ext = ort.InferenceSession(str(release_dir/'extractor.onnx'), sess_options=options, providers=['CPUExecutionProvider'])
clf = ort.InferenceSession(str(release_dir/'classifier.onnx'), sess_options=options, providers=['CPUExecutionProvider'])

def predict_onnx(path):
    image = rgb(path, cfg['cleaning']['max_pixels'])
    batch = transform(image).unsqueeze(0).numpy().astype('float32')
    features = ext.run(None, {ext.get_inputs()[0].name:batch})[0]
    labels, probabilities = clf.run(None, {clf.get_inputs()[0].name:features})
    return features, np.asarray(labels).reshape(-1), np.asarray(probabilities)

parity = []
for idx in splits[va].groupby('label').head(cfg['export']['parity_per_class']).index:
    features, labels, probabilities = predict_onnx(run.root/splits.loc[idx,'relative_path'])
    native_x = x[idx:idx+1]
    native_p = calibrated.predict_proba(native_x)
    parity.append({'image_id':splits.loc[idx,'image_id'],
                   'feature_max_error':float(np.abs(features-native_x).max()),
                   'probability_max_error':float(np.abs(probabilities-native_p).max()),
                   'passed':bool(np.array_equal(labels, calibrated.predict(native_x))
                                 and np.allclose(probabilities,native_p,atol=1e-4,rtol=1e-3))})
parity = pd.DataFrame(parity)
run.table(parity, 'onnx_calibrated_parity.csv')
if parity.empty or not parity.passed.all():
    raise ValueError('Parity ONNX gagal. Periksa onnx_calibrated_parity.csv.')

from sklearn.metrics import log_loss, top_k_accuracy_score
from kulitku_ml.evaluation import prediction_report
test_predictions = splits.loc[te, ['image_id','relative_path','label']].copy()
test_predictions['y_true'] = y[te]
test_predictions['y_pred'] = calibrated.predict(x[te])
probs = calibrated.predict_proba(x[te])
for index, label in enumerate(CLASSES):
    test_predictions[f'probability_{label}'] = probs[:,index]
run.table(test_predictions, 'test_predictions.csv')
prediction_report(run, test_predictions, 'test')
test_metrics = {**metrics(y[te], test_predictions.y_pred),
                'log_loss':log_loss(y[te],probs,labels=np.arange(4)),
                'top_2_accuracy':top_k_accuracy_score(y[te],probs,k=2,labels=np.arange(4)),
                'exploratory':True, 'patient_independence_verified':False}
run.json(test_metrics, 'test_metrics.json')
display(test_metrics)

prefix = f'onnx/{selected["combo_id"]}'
run.json({'classes':CLASSES, 'score_is_calibrated':True}, f'{prefix}/labels.json')
run.json({'transform':spec,'max_pixels':cfg['cleaning']['max_pixels'],
          'implementation':'timm.data.create_transform(is_training=False)',
          'rgb':'EXIF transpose; alpha composite white; RGB',
          'classifier_includes_scaler':name in ['logistic_regression','svm']}, f'{prefix}/preprocessing.json')
files_to_hash = ['extractor.onnx','classifier.onnx','labels.json','preprocessing.json']
run.json({'cnn':cnn,'classifier':name,'commit':commit,'calibration':'sigmoid_5fold_development',
          'parity_passed':True,'parity_samples':len(parity),'clinical_validation':False,
          'exploratory':True,'checksums':{file:file_hash(release_dir/file) for file in files_to_hash}},
          f'{prefix}/release.json')
bundle = shutil.make_archive(str(run.out/'kulitku_calibrated_onnx'), 'zip', root_dir=release_dir)
print('Paket ONNX:', bundle)


## Uji foto dari galeri (opsional)

Paket ONNX dapat diuji di Colab. Untuk FastAPI di branch ini, sesuaikan loader: engine saat ini memakai classifier joblib dan label lama. Paket baru menggunakan ONNX untuk kedua tahap, dengan urutan kelas dari `labels.json`.


In [ ]:
from google.colab import files
from PIL import Image
from io import BytesIO

for filename, content in files.upload().items():
    with tempfile.TemporaryDirectory() as directory:
        image_path = Path(directory)/Path(filename).name
        image_path.write_bytes(content)
        _, labels, probabilities = predict_onnx(image_path)
    preview = Image.open(BytesIO(content)).convert('RGB')
    preview.thumbnail((450,450))
    display(preview)
    print(filename, '→', CLASSES[int(labels[0])])
    display(pd.DataFrame({'kelas':CLASSES,'probabilitas_persen':100*probabilities[0]})
            .sort_values('probabilitas_persen',ascending=False).reset_index(drop=True))
